# Lab: Multiple layer Neural Network (MLP) 
This notebook has been prepared by Hsiu-Wen (Kelly) Chang from MINES ParisTech for the class machine learning

## Pytorch 

Pytorch is the most powerful open source deep learning framework. It is a replacement for NumPy (a python library for scientific computing) to use the power of GPUs and other accelerators. Also, the automatic differentiation library is useful to implement neural netorks. In this practical lession, you will first review the use of Numpy to solve a regression problem. Then the use of Pytorch to solve the same problem. Finally, you will practice how to create a complete class to solve a classification problem. 

## Learning objectives
 - Use Numpy to build a simple neural network fit a third order polynomial to sine function and how to replace it with Pytorch
 - Practice the use of neural network from sratch: architecture, activations
 - Practice the use of pytorch module to train an image classifier

**You have four practice questions for your own coding**

## How to use this version
Fill every block between `### YOUR CODE HERE ###` and `### END OF YOUR CODE ###`, following the comments above it, and run the cells in order: later cells use the names the comments ask for. Practices 1 to 4 are the larger exercises.

## 1. Solving a regression task all by ourself (Numpy): Educational purpose only!
----------------
Let us begin by examining how to solve a regression problem using a polynomial function and the simple gradient descent method to determine the coefficients.   
$$
f(x)=a+bx^{1}+cx^{2}+dx^{3}
$$

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import time

# Create random input and output data
x_true = np.linspace(-math.pi, math.pi, 2000)
y_true = np.sin(x_true)

# Split data into 80% train and 20% test
mask = np.random.rand(len(x_true)) < 0.8
x_train, y_train = x_true[mask], y_true[mask]
x_val, y_val = x_true[~mask], y_true[~mask]

# Randomly initialize weights
a = np.random.randn()
b = np.random.randn()
c = np.random.randn()
d = np.random.randn()

# Hyperparameters
learning_rate = 1e-6
epochs = 2000

for t in range(epochs):
    # Forward pass: compute y_pred = a + b x + c x^2 + d x^3 on x_train,
    # then `loss`, the SUM (not the mean) of the squared errors. Numpy only.
    ### YOUR CODE HERE ###

    ### END OF YOUR CODE ###

    if t % 200 == 99:
        print(t, loss)

    # Backprop by hand: compute grad_a, grad_b, grad_c, grad_d, the derivatives
    # of the loss with respect to each coefficient. Derive them on paper first
    # with the chain rule. Which term do all four have in common?
    ### YOUR CODE HERE ###

    ### END OF YOUR CODE ###

    # Update weights: one step of gradient descent on a, b, c and d
    ### YOUR CODE HERE ###

    ### END OF YOUR CODE ###

# Print final result
# Expected: the printed loss decreases at every line, final MSE below about 0.01
y_pred = a + b * x_train + c * x_train ** 2 + d * x_train ** 3
train_loss=np.square(y_pred - y_train).sum()/y_train.shape[0]
print(f'Result: y = {a} + {b} x + {c} x^2 + {d} x^3')
y_pred_val = a + b * x_val + c * x_val ** 2 + d * x_val ** 3
test_loss = np.square(y_pred_val - y_val).sum()/y_val.shape[0]

print(f'Train MSE error: {train_loss}.   Test MSE error:{test_loss}')

plt.plot(x_train, y_pred, 'g.',label='Predicted train')
plt.plot(x_val, y_pred_val, 'r*', label='Predicted test')
plt.plot(x_true,y_true,label='Actual')
plt.legend()
plt.show()

## 1.2 Tensor and autograd 
As this simple code demonstrates, we have to calculate the gradients and update the weights manually; we will now learn how to use PyTorch and a GPU to accelerate this computation process.

- `tensor` is an n-dimensional array that can keep track of a computational graph and gradients, which is important for deep learning. When ``x.requires_grad`` is true, PyTorch's autograd will track its gradients during forward pass. 
- `autograd` is a package that define a compuational graph while execute forward pass of a NN. Nodes in the graphe will be tensors and functions will be edges. Backprogating through this graph then allows us to easily compute gradients 

You should be cautious about ``requires_grad`` because you don't need it on when it is during test mode. And a common error is caused by not reset gradient after the weight update.

In [ ]:
import torch
dtype = torch.float
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# move input and output to the device
x = torch.tensor(x_train, device=device, dtype=dtype)
y = torch.tensor(y_train, device=device, dtype=dtype)

# Randomly initialize weights and requires the tracking of gradients
a = torch.randn((), dtype=dtype, requires_grad=True)
b = torch.randn((), dtype=dtype, requires_grad=True)
c = torch.randn((), dtype=dtype, requires_grad=True)
d = torch.randn((), dtype=dtype, requires_grad=True)

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute y_pred, then `loss` (sum of squared errors, a tensor)
    ### YOUR CODE HERE ###

    ### END OF YOUR CODE ###

    if t % 200 == 99:
        print(t, loss.item())

    # Backward pass, then the manual weight update:
    #   1. let autograd fill a.grad, b.grad, c.grad, d.grad
    #   2. update the four weights. Why must this happen inside torch.no_grad()?
    #   3. reset the gradients. What goes wrong at the next iteration otherwise?
    ### YOUR CODE HERE ###

    ### END OF YOUR CODE ###

# Final prediction and plotting
y_pred = a + b * x + c * x ** 2 + d * x ** 3
print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')


## 1.3 nn and optim from PyTorch

Now, let us move on to a modern method for accomplishing this simple task using PyTorch. 

 - ``nn`` module

Computational graphs for deeper networks need a high level of abstraction: layers. The ``nn`` package defines a set of Modules that are roughly equivalent to neural network layers. Each module receives input Tensors and computes output Tensors, and also holds internal state, such as Tensors containing learnable parameters. In addition, the nn package defines a set of useful loss functions that we can use directly. Another important note is the use of ``model.train()`` and ``model.eval()``. Some modules, like Dropout and BatchNorm, behave differently during training and testing; we can use these functions to run the built-in modules properly. See this [document](https://docs.pytorch.org/docs/stable/notes/autograd.html#locally-disable-grad-doc) for more details. In our simple model, this won't change the result yet. 

 - ``optim``

We can see that updating the weights of our models by manually mutating the tensors holding learnable parameters (some might not be learnable when we do transfer learning) with torch.no_grad(). The ``optim`` package provides us modern optimization algorithms such as AdaGrad, RMSProp, and Adam. Here we demonstrate the RMSprop algorithm. 

To further refine our code, we will start using the fit function, which includes the training procedure. This function reduces the need for repeated code in training. This is useful when you fine-tune hyperparameters.

In [ ]:
# continaul with the same example
# prepare input features: x_features has shape (N, 3), columns x, x^2, x^3
# (look up torch.stack and its `dim` argument)
### YOUR CODE HERE ###

### END OF YOUR CODE ###

# define model: a torch.nn.Sequential holding
#   - a linear layer from the 3 features to 1 output
#   - torch.nn.Flatten(0,1), which turns the output from (N, 1) into (N,) to match y
# by default, the newly-created module lives on CPU. It is needed to make sure the all the tensors and modules are on the same device
### YOUR CODE HERE ###

### END OF YOUR CODE ###

# define `loss_fn`: mean squared error, with reduction='sum' like the previous cells
# define `optimizer`: RMSprop over the parameters of the model, lr=1e-3
### YOUR CODE HERE ###

### END OF YOUR CODE ###

def fit(model, loss_fn, optimizer, x_features, y, epochs=2000):
    for t in range(epochs):
        # Forward pass: compute y_pred, then `loss`
        ### YOUR CODE HERE ###

        ### END OF YOUR CODE ###

        if t % 200 == 99:
            print(t, loss.item())

        # The three optimizer steps, in the right order: reset gradients,
        # backward pass, weight update
        ### YOUR CODE HERE ###

        ### END OF YOUR CODE ###

model.train()
fit(model, loss_fn, optimizer, x_features, y)

linear_layer=model[0]
print(f'Result: y = {linear_layer.bias.item()} + {linear_layer.weight[:, 0].item()} x + {linear_layer.weight[:, 1].item()} x^2 + {linear_layer.weight[:, 2].item()} x^3')

In [ ]:
x_val = torch.tensor(x_val, device=device, dtype=dtype)
y_val = torch.tensor(y_val, device=device, dtype=dtype)

model.eval()
# Compute `pred_val`, the model output on the validation data. The model expects
# the same 3 features as during training.
### YOUR CODE HERE ###

### END OF YOUR CODE ###
print(f'Validation loss:{loss_fn(pred_val,y_val).item()}')

 - Wrape all you need to a class

In deep learning, it is important to test several models with different hyperparameter settings. Here is how we can wrap what we need to a class.

In [ ]:
class mlpregression(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Declare the two layers of the Sequential model above as attributes:
        # self.lc (the linear layer) and self.fc (the Flatten)
        ### YOUR CODE HERE ###

        ### END OF YOUR CODE ###

    def forward(self, x):
        # Pass x through both layers and store the output in y
        ### YOUR CODE HERE ###

        ### END OF YOUR CODE ###
        return y
    

$\textbf{Practice 1}$: use this class to redo the training of the regression task. Overall, you should have the flowchat of the algorithm like this
1. initialization
2. forward
3. loss
4. Backward
5. optimize
6. print the test loss

In [ ]:
# Use a NEW model called `mymodel`, a NEW optimizer built on its parameters, and
# the `fit` function defined above. Careful in the last step: the variable
# `model` still exists from the previous section, make sure you evaluate `mymodel`.

### YOUR CODE HERE ###

### END OF YOUR CODE ###


## 2. Pytorch on classifiction with hand-drawn digits images
----------------

We are going to targeting the application of predicting the hand-drawn digits (between 0 and 9) in a given black-and-white image. First, we will see how to preprocess the data and eventually use the public dataset provided by ```torchvision```. 

### 2.1 download raw images by URL. 
![](http://perso.mines-paristech.fr/fabien.moutarde/ES_MachineLearning/TP_convNets/mnist.png)  
You can use [pathlib](https://docs.python.org/3/library/pathlib.html)
for dealing with paths (part of the Python 3 standard library), and download the dataset using
[requests](http://docs.python-requests.org/en/master/). We will only import modules when we use them, so you can see exactly what's being used at each point.


In [ ]:
from pathlib import Path
import requests

DATA_PATH = Path("data")
PATH = DATA_PATH / "mnist"

PATH.mkdir(parents=True, exist_ok=True)

URL = "https://github.com/pytorch/tutorials/raw/main/_static/"
FILENAME = "mnist.pkl.gz"

if not (PATH / FILENAME).exists():
        content = requests.get(URL + FILENAME).content
        (PATH / FILENAME).open("wb").write(content)

In [ ]:
import os
print(os.getcwd())
print((PATH / FILENAME))

This dataset is in numpy array format, and has been stored using pickle, a python-specific format for serializing data. Each image is 28x28 pixels, and is being stored as a flattened rwo of length 784. To have a look at one provided image, we need to reshape it to 2D. In addition, PyTorch uses ``torch.tensor``, rather than numpy arrays, so we need to convert our data.

In [ ]:
import pickle
import gzip

with gzip.open((PATH / FILENAME).as_posix(), "rb") as f:
        ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding="latin-1")

plt.imshow(x_train[0].reshape(28,28), cmap='gray')
print(f"size of training data (number of samples, number of pixels): {x_train.shape}")
print(f"size of test data (number of samples, number of pixels): {x_valid.shape}")

# Convert data to torch tensors and move to detected device
x_train, y_train, x_valid, y_valid = map(
    lambda x: torch.tensor(x).to(device), (x_train, y_train, x_valid, y_valid)
)

## 2.2 Without nn modules

In [ ]:
# Create a NN with linear model and (28*28) inputs and 10 outputs

# Initialize `weights` and `bias` with torch.randn. Both must be learnable and
# live on `device`, with type `dtype`. Which shapes do they need so that
# (batch, 784) @ weights + bias gives (batch, 10)?
### YOUR CODE HERE ###

### END OF YOUR CODE ###

Pytorch has several built-in activation functions in ```torch.nn.functional```, however, it is necessary to know how to define one by yourself. PyTorch will
even create fast GPU or vectorized CPU code for your function automatically. Let's define log_softmax function and use it as our activation to increase the complexity of our NN

$${\bf \theta} = \log\left(\frac{{\exp}({x_{i})}}{\sum_{j}{\bf \exp}({x_{j})}}\right)$$
$${\bf \theta} = x-\log{\sum_{j}{\bf \exp}({x_{j})}}$$

In [ ]:
def log_softmax(x):
    # Implement the second formula above and store the result in log_probs.
    # x has shape (batch, 10): the sum runs over the classes, not the batch.
    # torch.logsumexp avoids computing log(exp(...)) by hand. Why is that safer?
    ### YOUR CODE HERE ###

    ### END OF YOUR CODE ###
    return log_probs

def model(x):
    # Implement the NN: linear transformation of x, then log_softmax
    ### YOUR CODE HERE ###

    ### END OF YOUR CODE ###
    return log_preds

In the above, the ``@`` stands for the dot product operation.

Now we design a batch size of 64 to train each time

In [ ]:
bs = 64  # batch size

xb = x_train[0:bs]  # a mini-batch from x without randomness

preds = model(xb)  # predictions with initial model

print(preds[0], preds.shape)

As you see, the ``preds`` tensor contains not only the tensor values, but also a
gradient function. We'll use this later to do backprop.

Let's implement negative log-likelihood to use as the loss function
(again, we can just use standard Python):



In [ ]:
def nll(input, target):
    # input has shape (batch, 10) and holds log-probabilities, target has shape
    # (batch,) and holds the true class indices. For each sample, keep only the
    # log-probability of its true class, then return minus the mean.
    # No loop needed: think about indexing with two sequences at once.
    ### YOUR CODE HERE ###

    ### END OF YOUR CODE ###
    return loss

loss_func = nll

Let's check our loss with our random model, so we can see if we improve
after a backprop pass later.



In [ ]:
yb = y_train[0:bs]
print(preds[range(yb.shape[0]),yb])
print(f'Current loss:{loss_func(preds, yb).item()}')

Let's also implement a function to calculate the accuracy of our model.
For each prediction, if the index with the largest value matches the
target value, then the prediction was correct.



In [ ]:
def accuracy(out, yb):
    # 1. the predicted class of a sample is the index of its largest output
    # 2. compare with yb and average. Comparing gives booleans, what must you
    #    convert them to before taking a mean?
    ### YOUR CODE HERE ###

    ### END OF YOUR CODE ###
    return acc

Let's check the accuracy of our random model, so we can see if our
accuracy improves as our loss improves.



In [ ]:
print(accuracy(preds, yb))

We can now run a training loop.  For each iteration, we will:

- select a mini-batch of data (of size ``bs``)
- use the model to make predictions
- calculate the loss
- ``loss.backward()`` updates the gradients of the model, in this case, ``weights``
  and ``bias``.

In [ ]:
lr = 0.5  # learning rate
epochs = 2  # how many epochs to train for

for epoch in range(epochs):
    for i in range((x_train.shape[0] - 1) // bs + 1): #floor division
        start_i=i*bs
        end_i =start_i + bs
        # grab batch of data (xb) and targets (yb)
        xb = x_train[start_i:end_i]
        yb = y_train[start_i:end_i]

        # Train on this batch:
        #   1. predict, then compute `loss` with loss_func
        #   2. backpropagate
        #   3. inside torch.no_grad(): update `weights` and `bias`, then zero
        #      their gradients in place
        ### YOUR CODE HERE ###

        ### END OF YOUR CODE ###

    # Expected: the loss falls well below its starting value of about 10
    print('Epochs {}: loss {}, accuracy {}'.format(epoch, loss_func(model(xb), yb), accuracy(model(xb), yb)))

That's it: we've created and trained a minimal neural network (in this case, a
logistic regression, since we have no hidden layers) entirely from scratch!

Let's check the loss and accuracy and compare those to what we got
earlier. We expect that the loss will have decreased and accuracy to
have increased, and they have.



In [ ]:
print('loss:', loss_func(model(xb), yb).item(), 'accuracy:', accuracy(model(xb), yb).item())

### 2.3 Using PyTorch modules for training, testing and validation


The above work can be easily done by using torch.nn. Here is how we can code it in a pretty way. 

Here is the list of modules that you should try to use:
1. [torch.nn](https://pytorch.org/docs/stable/nn.html): Linear, CrossEntropyLoss, functional.log_softmax
2. [torch.optim](https://pytorch.org/docs/stable/optim.html): SGD
3. [Dataset](https://pytorch.org/docs/stable/data.html?highlight=dataset#torch.utils.data.Dataset)
4. [DataLoader](https://pytorch.org/docs/stable/data.html?highlight=dataloader#torch.utils.data.DataLoader)

In [ ]:
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim

# Step 1: use activation and loss function from ``torch.nn.functional``. We use torch.nn.Parameter for defining learnable parameters of a model. They can be returned by Minist_Logistic.parameters()
# Define your model using nn.Module
class Mnist_Logistic(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # One linear layer from the 784 pixels to the 10 classes. Name it
        # self.fc1: the next cell reads model.fc1.
        ### YOUR CODE HERE ###

        ### END OF YOUR CODE ###

    def forward(self, x):
        # Linear layer, then F.log_softmax. Over which dimension?
        ### YOUR CODE HERE ###

        ### END OF YOUR CODE ###
        return log_probs

# Intialize the model and move it to the device
model = Mnist_Logistic().to(device)
print(model)
# see the size of learnable parameters
params=list(model.parameters())
print(params[0].shape)  # weights
print(params[1].shape)  # bias

# Step 2: Define a loss function and an optimizer
# `criterion`: cross-entropy from torch.nn
### YOUR CODE HERE ###

### END OF YOUR CODE ###

out=model(x_train[0].unsqueeze(0))  # unsqueeze to add batch dimension
print(f'Dimenstion of output tensor:{out.shape}')
loss=criterion(out, y_train[0].unsqueeze(0))

# Step 3: Define a optimizer that uses stochastic gradient descent to update trainable parameters
# `optimizer`: SGD over the parameters of the model, lr=0.1
### YOUR CODE HERE ###

### END OF YOUR CODE ###


In [ ]:
print(loss)
# For illustration, let us follow a few steps backward
print(loss.grad_fn)  # NLLLoss
print(loss.grad_fn.next_functions[0][0])  # LogSoftmax

model.zero_grad()
print('fc1 weight grad before backward:', model.fc1.weight.grad)

loss.backward()
print('fc1 weight grad after backward:', model.fc1.weight.grad)

In [ ]:
def fit(model, loss_func, optimizer, x_train, y_train, bs, epochs=5):
    for epoch in range(epochs):
        for i in range((x_train.shape[0] - 1) // bs + 1): #floor division
            start_i=i*bs
            end_i =start_i + bs
            # grab batch of data (xb) and targets (yb)
            xb = x_train[start_i:end_i]
            yb = y_train[start_i:end_i]

            # Same training step as before, but the optimizer now does the
            # update and the zeroing for you. Which manual lines disappear?
            ### YOUR CODE HERE ###

            ### END OF YOUR CODE ###

        print('Epochs {}: loss {}, accuracy {}'.format(epoch, loss_func(model(xb), yb), accuracy(model(xb), yb)))

Instantiate your function and check loss value



In [ ]:
fit(model, criterion, optimizer, x_train, y_train, bs, epochs=5)

$\textbf{Practice 2}$: Add one more layer to improve the accuracy

In [ ]:
# Write class Mnist_MLP(nn.Module) with one hidden layer:
#     Linear(784 -> hidden) -> ReLU -> Linear(hidden -> 10)
# Start with hidden = 128. forward() can return the raw scores of the last layer,
# because nn.CrossEntropyLoss applies log_softmax itself.
#
# Then instantiate it on `device`, build a new SGD optimizer and call `fit`.
# Compare with the accuracy of Mnist_Logistic above. Note that `fit` only prints
# the loss and accuracy of the LAST batch: Practice 3 gives a more reliable measure.

### YOUR CODE HERE ###

### END OF YOUR CODE ###


Until now, we can see we are not smart in picking a batch of input. It is better to you TensorDataset tools that provide us the randonness and split function to deal with data sampling. 

PyTorch has an abstract Dataset class. A Dataset can be anything that has a __len__ function (called by Python's standard len function) and a __getitem__ function as a way of indexing into it. This [tutorial](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html) walks through a nice example of creating a custom FacialLandmarkDataset class as a subclass of Dataset.

By defining a length and way of indexing, this also gives us a way to iterate, index, and slice along the first dimension of a tensor. This will make it easier to access both the independent and dependent variables in the same line as we train.

In [ ]:
from torch.utils.data import TensorDataset
train_ds = TensorDataset(x_train, y_train)

# This is how to get a batch of data from the TensorDataset
xb, yb = train_ds[0:bs]

print(xb.shape, yb.shape)


- using DataLoader

Pytorch's ``DataLoader`` is responsible for managing batches. You can
create a ``DataLoader`` from any ``Dataset``. ``DataLoader`` makes it easier
to iterate over batches. Rather than having to use ``train_ds[i*bs : i*bs+bs]``,
the DataLoader gives us each minibatch automatically.



In [ ]:
from torch.utils.data import DataLoader

train_dl = DataLoader(train_ds, batch_size=bs)

In [ ]:
# use the DataLoader to get a batch of data
xb, yb = next(iter(train_dl)) 
print(xb.shape, yb.shape)

Thanks to Pytorch's ``nn.Module``, ``nn.Parameter``, ``Dataset``, and ``DataLoader``,
our training loop is now dramatically smaller and easier to understand. Let's
now try to add the basic features necessary to create effecive models in practice.

 -  Add validation
In reality, we always need to take care the overfitting issue. We can avoid it by:
(1) add validation
(2) shuffling the training data prevent correlation between batches and overfitting. 

In [ ]:
# Create `train_ds` and `valid_ds` with TensorDataset, then:
#   - `train_dl`: batch size bs, shuffled
#   - `valid_dl`: batch size bs * 2, not shuffled
# Why does validation not need shuffling? Why can its batches be larger?
### YOUR CODE HERE ###

### END OF YOUR CODE ###

(Note that we always call ``model.train()`` before training, and ``model.eval()``
before inference, because these are used by layers such as ``nn.BatchNorm2d``
and ``nn.Dropout`` to ensure appropriate behaviour for these different phases.)

$\textbf{Practice 3}$: rewrite the fit function with train_dl and valid_dl and save the loss and accuracy for each epoch in order to to practice 4.

In [ ]:
# Write fit_dl(model, loss_func, optimizer, train_dl, valid_dl, epochs=5) that
# returns `history`, a dict of 4 lists with one value per epoch:
#     history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
#
# For each epoch:
#   - model.train(), then loop over train_dl and do the training step
#   - model.eval(), then loop over valid_dl under torch.no_grad(), no weight update
#   - append the epoch's values to history and print them
#
# Careful with the averages: the last batch of a loader is smaller than the others,
# so a plain mean of the batch values is slightly wrong. Weight each batch by
# xb.shape[0], and use loss.item() so you do not keep the whole graph in memory.
#
# Then train a new Mnist_MLP (not `mlp`, which is already trained) and keep the
# result in `history`.

### YOUR CODE HERE ###

### END OF YOUR CODE ###


$\textbf{Practice 4}$: plot loss values and validations values
In order to understand the progress of your training, error curves with test and validation are very common to be shown.
Change your code in order to see the curves

In [ ]:
# Using `history` from Practice 3, draw two plots side by side (plt.subplots):
#   - left: train and validation loss per epoch
#   - right: train and validation accuracy per epoch
# With axis labels and a legend. Then rerun Practice 3 with more epochs:
# from which epoch does the validation curve stop following the training curve?

### YOUR CODE HERE ###

### END OF YOUR CODE ###

Summary 
-----------------

We have seen and practice the following functions

 - **torch.nn**

   + ``Module``: creates a callable which behaves like a function, but can also
     contain state(such as neural net layer weights). It knows what ``Parameter`` (s) it
     contains and can zero all their gradients, loop through them for weight updates, etc.
   + ``Parameter``: a wrapper for a tensor that tells a ``Module`` that it has weights
     that need updating during backprop. Only tensors with the `requires_grad` attribute set are updated
   + ``functional``: a module(usually imported into the ``F`` namespace by convention)
     which contains activation functions, loss functions, etc, as well as non-stateful
     versions of layers such as convolutional and linear layers.
 - ``torch.optim``: Contains optimizers such as ``SGD``, which update the weights
   of ``Parameter`` during the backward step
 - ``Dataset``: An abstract interface of objects with a ``__len__`` and a ``__getitem__``,
   including classes provided with Pytorch such as ``TensorDataset``
 - ``DataLoader``: Takes any ``Dataset`` and creates an iterator which returns batches of data.

